# Übung 07: EDA – nicht-grafisch

**Übungsnotebook zur Vorbereitung auf LN 1 (Mo. 09.11.2026)** · Cheatsheet: **Kapitel 7 (EDA – nicht-grafisch) und 9 (Statistik-Grundlagen)** und Kapitel 11 (Checklisten)

| | |
|---|---|
| **Themen** | Systematik der EDA, Lage-/Streuungs-/Formmasse, Häufigkeiten, Kreuztabellen, Kovarianz & Korrelation (Pearson/Spearman), Ausreisser, Chi²-Test, Interpretation |
| **Format** | wie die Musterprüfungen AP01–AP03: Teil A Konzeptfragen · Teil B Fehleranalyse · Teil C Code schreiben |
| **Zeitbudget** | ca. 230 Minuten (Prüfungstempo: ca. 4.5 Min. pro Punkt) |
| **Lösungen** | keine – alles ist mit dem Cheatsheet bzw. den Kursunterlagen lösbar. Hinweise wie *(→ 4.5)* verweisen auf den Cheatsheet-Abschnitt. |

**Szenario:** Der fiktive E-Bike-Verleih *VeloWin* betreibt 8 Stationen in Winterthur. Sie erhalten eine Stichprobe von 5'000 Fahrten (Juni–September 2026) und die täglichen Fahrtenzahlen mit Wetterdaten. Das Management will wissen, wie die Fahrten «typischerweise» aussehen und welche Faktoren die Nachfrage beeinflussen.

**So üben Sie prüfungsnah**
- Copilot/KI-Autocomplete ausschalten, wie in der Prüfung (Cheatsheet 1.1).
- Begründungen immer **mit konkretem Bezug zu den Daten** (Spaltennamen, Werte, Zahlen aus dem Output) – sonst gibt es in der Prüfung 0 Punkte.
- Verändern Sie die Ausgangsdaten nicht, sondern arbeiten Sie pro Aufgabe mit eigenen Variablen (z.B. `df_B1`, `df_C2`).
- Am Schluss: *Kernel → Restart & Run All* (Cheatsheet 11.5).

In diesem Notebook sind **keine Grafiken** verlangt (die folgen in Übung 08). Sie dürfen zur Kontrolle aber jederzeit eine Grafik erstellen.

## Setup (wird nicht bewertet)

Die folgende Zelle erzeugt die Übungsdaten **synthetisch und reproduzierbar** (fixer Seed) im Unterordner neben diesem Notebook. Sie brauchen den Code nicht zu verstehen und sollen ihn **nicht verändern** – behandeln Sie die erzeugten Dateien so, als hätten Sie sie in der Prüfung erhalten. *(Natürlich dürfen Sie die Dateien im Editor öffnen und ansehen.)*

In [ ]:
# Libraries
import os
import numpy as np
import pandas as pd
from scipy import stats

# Settings
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Get current working directory
print(os.getcwd())

In [ ]:
# ---------------------------------------------------------------
# Data generator (do not modify) -> folder 'data_uebung07'
# ---------------------------------------------------------------
import os
import numpy as np
import pandas as pd

rng = np.random.default_rng(707)
D = 'data_uebung07'
os.makedirs(D, exist_ok=True)

stations = ['Bahnhof HB', 'Technikum', 'Altstadt', 'Oberwinterthur', 'Töss', 'Wülflingen', 'Seen', 'Rosenberg']
st_w = np.array([0.24, 0.16, 0.14, 0.12, 0.10, 0.09, 0.08, 0.07])
tourist_p = {'Bahnhof HB': 0.20, 'Altstadt': 0.22, 'Technikum': 0.03, 'Oberwinterthur': 0.05,
             'Töss': 0.04, 'Wülflingen': 0.05, 'Seen': 0.06, 'Rosenberg': 0.08}

# daily weather and number of rides (inverted U in temperature)
days = pd.date_range('2026-06-01', '2026-09-30', freq='D')
temp = np.round(19 + 7 * np.sin((days.dayofyear - 140) / 365 * 2 * np.pi) + rng.normal(0, 4.5, len(days)), 1)
rain = np.round(np.where(rng.random(len(days)) < 0.3, rng.gamma(1.5, 5, len(days)), 0), 1)
mu = 260 - 1.9 * (temp - np.mean(temp)) ** 2 - 4.5 * rain + rng.normal(0, 18, len(days))
daily = pd.DataFrame({'date': days.strftime('%Y-%m-%d'), 'temp_c': temp, 'rain_mm': rain,
                      'n_rides': np.clip(np.round(mu), 15, None).astype(int)})
daily.to_csv(f'{D}/daily.csv', index=False)

# individual rides (a sample of 5000 rides)
n = 5000
st = rng.choice(stations, size=n, p=st_w)
ut = np.array([rng.choice(['Abo', 'Einzel', 'Tourist'], p=[0.7 - tourist_p[s] * 0.5, 0.3 - tourist_p[s] * 0.5, tourist_p[s]]) for s in st])
speed = np.where(ut == 'Tourist', rng.normal(10, 2, n), np.where(ut == 'Abo', rng.normal(17, 2.5, n), rng.normal(14.5, 2.5, n)))
speed = np.clip(speed, 5, 25)
dist = np.round(rng.gamma(2.0, 1.4, n) + 0.3, 2)
stop = np.where(ut == 'Tourist', rng.exponential(18, n), rng.exponential(2.5, n))
dur = np.round(dist / speed * 60 + stop + rng.normal(0, 1.5, n).clip(-1, None), 2)
dur = np.clip(dur, 1.0, None)
forgot = rng.choice(n, size=7, replace=False)
dur[forgot] = np.round(rng.uniform(1300, 1440, 7), 2)        # bike not locked properly -> ride runs on
rating = np.where(rng.random(n) < 0.42, np.nan,
                  np.clip(np.round(4.1 - 0.012 * dur.clip(None, 120) + rng.normal(0, 0.8, n)), 1, 5))
rating = np.where(ut == 'Tourist', np.where(rng.random(n) < 0.7, np.nan, rating), rating)
battery = np.clip(np.round(rng.normal(72, 18, n)), 12, 100)
ride_day = rng.choice(len(days), size=n, p=daily['n_rides'] / daily['n_rides'].sum())
start = days[ride_day] + pd.to_timedelta(rng.integers(6 * 60, 22 * 60, n), unit='min')
rides = pd.DataFrame({'ride_id': [f'R{100000 + i}' for i in range(n)], 'start_time': start.strftime('%Y-%m-%d %H:%M'),
                      'start_station': st, 'user_type': ut, 'distance_km': dist, 'duration_min': dur,
                      'battery_start_pct': battery, 'temp_c': temp[ride_day], 'rating': rating})
rides.to_csv(f'{D}/rides.csv', index=False)
print('Files created:', sorted(os.listdir(D)))

### Daten laden

| Datei | Spalten |
|---|---|
| `rides.csv` | `ride_id`, `start_time`, `start_station`, `user_type` (`Abo`, `Einzel`, `Tourist`), `distance_km`, `duration_min`, `battery_start_pct`, `temp_c` (Tagestemperatur), `rating` (1–5 Sterne, freiwillig) |
| `daily.csv` | `date`, `temp_c` (Tagesmittel), `rain_mm`, `n_rides` (alle Fahrten des Tages) |

In [ ]:
# Load the data (do not modify df and df_daily)
df = pd.read_csv('data_uebung07/rides.csv')
df_daily = pd.read_csv('data_uebung07/daily.csv')

print('rides:', df.shape, '| daily:', df_daily.shape)
df.head()

---

<div style="background-color:#2E4057; color:white; padding:16px 24px; border-radius:6px; font-size:18px; margin-top:8px;">
&#9654;&nbsp; <strong>Ab hier beginnen die Übungsaufgaben — 56 Punkte</strong><br>
<span style="font-size:15px; font-weight:normal;">Tragen Sie Ihre Antworten direkt in die vorgesehenen Zellen ein und führen Sie alle Code-Zellen aus.</span>
</div>

---
## Teil A – Konzeptfragen (12 Punkte)

*Antworten ohne Bezug zu den VeloWin-Daten (Spaltennamen, konkrete Zahlen) werden nicht gewertet.*

### A1 (4 Punkte)

Führen Sie `df['duration_min'].describe()` und `df['duration_min'].skew()` aus.

Beschreiben Sie die Verteilung der Fahrtdauer anhand dieser Kennzahlen (Lage, Streuung, Form, Auffälligkeiten). Welches Lagemass berichten Sie dem Management als «typische Fahrtdauer», und warum? Was fällt Ihnen beim Maximum auf, und was prüfen Sie als Nächstes? *(→ 7.2, 9.2)*

In [ ]:
# Code hier eintragen

**Ihre Antwort:**

> [Hier eintragen]

### A2 (4 Punkte)

Bestimmen Sie für die Variablen `user_type`, `start_station`, `rating`, `duration_min`, `temp_c` und `battery_start_pct` das **Skalenniveau** und je ein **geeignetes Lagemass**. Welche Korrelationsmethode verwenden Sie für den Zusammenhang zwischen `rating` und `duration_min`, und warum? *(→ 9.1, 7.3)*

**Ihre Antwort:**

> [Hier eintragen]

### A3 (4 Punkte)

Ein Kollege berechnet mit `df_daily` die Pearson-Korrelation zwischen `temp_c` und `n_rides`, erhält r ≈ 0.2 und schreibt: *«Die Temperatur hat höchstens einen schwachen Einfluss auf die Nachfrage – je wärmer, desto etwas mehr Fahrten.»*

Prüfen Sie die Aussage **nicht-grafisch**: Teilen Sie die Temperatur mit `pd.cut` in Klassen (z.B. 5-Grad-Schritte) und berechnen Sie pro Klasse Anzahl Tage und mittlere bzw. mediane Anzahl Fahrten. Was zeigt sich? Nennen Sie **zwei** fachliche Einwände gegen die Aussage des Kollegen. *(→ 7.3, 9.5)*

In [ ]:
# Code hier eintragen

**Ihre Antwort:**

> [Hier eintragen]

---
## Teil B – Fehleranalyse Python-Code (24 Punkte)

*Jeder Code läuft fehlerfrei durch, liefert aber ein falsches oder irreführendes Ergebnis.*

### B1 (8 Punkte)

**Ziel:** Wie stark hängt die **Dauer einer Fahrt** mit ihrer **Distanz** zusammen? (Analyseeinheit: die einzelne Fahrt.)

In [ ]:
# Original code (do not modify this cell)
df_B1_orig = df.groupby('start_station')[['distance_km', 'duration_min']].mean()
r = df_B1_orig['distance_km'].corr(df_B1_orig['duration_min'])

print('Pearson r:', round(r, 2))
print(f'Fazit: Die Distanz erklärt {r**2:.0%} der Variation der Fahrtdauer.')

**Aufgabe:** Erklären Sie, warum Ergebnis und Fazit irreführend sind (mindestens **zwei** Gründe, Tipp: Analyseeinheit, Ausreisser, Schiefe). Schreiben Sie den korrigierten Code, der den Zusammenhang auf der richtigen Ebene mit einer geeigneten Methode misst (Pearson **und** Spearman, mit und ohne die unplausiblen Fahrten), und begründen Sie jede Änderung. Interpretieren Sie das korrigierte Ergebnis in 1–2 Sätzen. *(→ 7.3, 9.5)*

**Begründung der Änderungen:**

> [Hier eintragen]

**Korrigierter Code:**

In [ ]:
# Korrigierten Code unter den Kommentaren eintragen

### B2 (8 Punkte)

**Ziel:** Für die Planung der Velo-Verteilung soll pro **Station** angegeben werden, wie sich die Fahrten auf die Nutzertypen verteilen (Anteile in %, jede Station = 100 %).

In [ ]:
# Original code (do not modify this cell)
df_B2_orig = (pd.crosstab(df['start_station'], df['user_type'], normalize='columns') * 100).round(1)
print(df_B2_orig)
print('\nFazit: Am Bahnhof HB sind', df_B2_orig.loc['Bahnhof HB', 'Tourist'], '% der Fahrten von Touristen.')

**Aufgabe:** Erklären Sie, was die Prozentwerte im Original tatsächlich bedeuten und warum das Fazit falsch ist. Schreiben Sie den korrigierten Code (inkl. Zeile/Spalte `Total` und einer Kontrolle, dass jede Station 100 % ergibt) und formulieren Sie das korrekte Fazit für den Bahnhof HB. Begründen Sie jede Änderung. *(→ 6.9, 7.3)*

**Begründung der Änderungen:**

> [Hier eintragen]

**Korrigierter Code:**

In [ ]:
# Korrigierten Code unter den Kommentaren eintragen

### B3 (8 Punkte)

**Ziel:** Steckbrief «typische Fahrt» **pro Nutzertyp**: typische Dauer, robuste Streuung und Anteil der Fahrten über 60 Minuten.

In [ ]:
# Original code (do not modify this cell)
rows = []
for ut, g in df.groupby('user_type'):
    rows.append({'user_type': ut,
                 'typical_duration': g['duration_min'].mode()[0],
                 'spread': g['duration_min'].std(),
                 'share_over_60_pct': round((df['duration_min'] > 60).sum() / len(df) * 100, 1)})
df_B3_orig = pd.DataFrame(rows)
df_B3_orig

**Aufgabe:** Finden Sie **drei** Fehler bzw. ungeeignete Kennzahlen (Tipp: `g['duration_min'].nunique()` und `len(g['duration_min'].mode())`). Schreiben Sie den korrigierten Steckbrief (zusätzlich mit `n_rides`) und begründen Sie jede Änderung. *(→ 7.2, 9.2, 6.6)*

**Begründung der Änderungen:**

> [Hier eintragen]

**Korrigierter Code:**

In [ ]:
# Korrigierten Code unter den Kommentaren eintragen

---
## Teil C – Python-Code schreiben (20 Punkte)

*Schreiben Sie Python-Code und begründen bzw. interpretieren Sie jeweils in 1–2 Sätzen.*

### C1 (4 Punkte)

Erstellen Sie für die numerischen Variablen `distance_km`, `duration_min`, `battery_start_pct` und `temp_c` **eine** Kennzahlentabelle (Variablen als Zeilen) mit: `count`, `mean`, `median`, `std`, `iqr`, `cv` (Variationskoeffizient), `skew`, `kurtosis` – gerundet auf 2 Stellen. Tipp: `agg()` mit einer Liste aus Funktionsnamen und eigenen Funktionen, danach `.T`.

Welche Variable ist am stärksten schief, welche streut **relativ** am stärksten? Begründen Sie, warum Sie für den Vergleich der Streuung den CV und nicht die Standardabweichung verwenden. *(→ 7.2, 9.2)*

**Begründung und Interpretation:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

### C2 (4 Punkte)

Erstellen Sie eine **Häufigkeitstabelle** für `rating` mit den Spalten `n`, `pct` und `cum_pct` (1 Stelle). Fehlende Bewertungen sollen als eigene Kategorie `keine Bewertung` erscheinen; die Sterne sollen aufsteigend sortiert sein.

Vergleichen Sie danach den Anteil fehlender Bewertungen **pro Nutzertyp**. Was bedeutet das für die Aussage «Die durchschnittliche Bewertung aller Fahrten beträgt x Sterne»? *(→ 6.4, 7.2, 9.4)*

**Begründung und Interpretation:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

### C3 (4 Punkte)

Erstellen Sie eine Kreuztabelle **Nutzertyp × Rating** (nur Fahrten mit Bewertung) – einmal mit absoluten Häufigkeiten inkl. Totalen, einmal mit Zeilenanteilen in %. Prüfen Sie mit `stats.chi2_contingency` (auf den **absoluten** Häufigkeiten **ohne** Totale), ob Nutzertyp und Bewertung unabhängig sind. Geben Sie Chi², Freiheitsgrade und p-Wert aus und interpretieren Sie das Ergebnis. *(→ 7.3, 9.6)*

**Begründung und Interpretation:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

### C4 (4 Punkte)

Berechnen Sie für die numerischen Spalten von `df` (ohne `rating`) die Pearson- **und** die Spearman-Korrelationsmatrix (2 Stellen). Bestimmen Sie programmatisch das Variablenpaar mit dem **grössten Unterschied** `|r − ρ|`. Erklären Sie diesen Unterschied mit den Eigenschaften der beteiligten Variablen. *(→ 7.3, 9.5)*

**Begründung und Interpretation:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

### C5 (4 Punkte)

Kennzeichnen Sie Ausreisser der Fahrtdauer **innerhalb jedes Nutzertyps** nach der Tukey-Regel (1.5 × IQR, Grenzen pro Gruppe mit `groupby(...).transform(...)`) sowie nach der z-Wert-Regel (|z| > 3, ebenfalls pro Gruppe). Geben Sie pro Nutzertyp die Anzahl Ausreisser nach beiden Regeln aus.

Warum unterscheiden sich die Ergebnisse der beiden Regeln, und welche Regel ist für diese Daten besser geeignet? Welche der Ausreisser sind vermutlich Datenfehler? *(→ 6.5, 9.2, 9.3)*

**Begründung und Interpretation:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

---
### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Libraries
import os, platform, socket
from platform import python_version
from datetime import datetime

# Print system information
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print('Python Version:', python_version())
print('-----------------------------------')